# 🔬 Phase 3B: Test Evaluation — 6-Way Comparison (max=80 tokens)

**Mục tiêu:** Đánh giá trên 500 mẫu Test với 6 điều kiện:  
1. Vanilla Baseline  
2. 🆕 Best Cosine Decay Early-Stop (từ 3A)  
3. 🆕 Runner-up Early-Stop (từ 3A)  
4. Full Steering (α=20, K=∞)  
5. Control: Random Direction  
6. Control: Sign-Flipped  

**Metrics:** ROUGE-L + BERTScore + 4-gram Repetition + EOS Rate + Category Breakdown + Bootstrap p-value  
**Thời gian ước tính:** ~9 giờ  
**Input:** Phase 1 artifacts + Dataset 15K  
**Output:** `phase3b_test80_results.json`

---

In [ ]:
# Cell 1: Install
!pip install -q bitsandbytes accelerate transformers torch rouge-score bert-score tqdm
print('✅ Done')

In [ ]:
# Cell 2: Imports
import os, json, glob, random, time, math, gc
import numpy as np
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR = '/kaggle/working'

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  ✅ CẤU HÌNH TỐI ƯU TỪ PHASE 3A (TỰ ĐỘNG ĐÃ LẮP VÀO)         ║
# ╚══════════════════════════════════════════════════════════════╝
BEST_ES_ALPHA = 18.0       # ← Tối ưu nhất từ Phase 3A
BEST_ES_K = 16             # ← Tối ưu nhất từ Phase 3A
BEST_ES_DECAY = 'linear'   # ← Tối ưu nhất từ Phase 3A

RUNNER_ES_ALPHA = 15.0     # ← Runner-up từ Phase 3A
RUNNER_ES_K = 16           # ← Runner-up từ Phase 3A
RUNNER_ES_DECAY = 'linear' # ← Runner-up từ Phase 3A

print(f'Best ES config:    α={BEST_ES_ALPHA}, K={BEST_ES_K}, decay={BEST_ES_DECAY}')
print(f'Runner-up config:  α={RUNNER_ES_ALPHA}, K={RUNNER_ES_K}, decay={RUNNER_ES_DECAY}')


In [ ]:
# Cell 4: Data Loading (IDENTICAL split)
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}', f'./{DATA_FILENAME}'
]
data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches: data_path = matches[0]; break
if not data_path: raise FileNotFoundError(f'❌ {DATA_FILENAME} not found')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)
shuffled_records = list(raw_dataset)
random.seed(SEED)
random.shuffle(shuffled_records)
n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
test_records = shuffled_records[n_train + n_val:]
print(f'📊 Test: {len(test_records):,} records')

In [ ]:
# Cell 5: Load Phase 1 Artifacts
config_paths = glob.glob('/kaggle/input/**/steering_config.json', recursive=True)
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
v_rand_paths = glob.glob('/kaggle/input/**/v_rand.pt', recursive=True)
if not config_paths: raise FileNotFoundError('❌ steering_config.json not found')

with open(config_paths[0], 'r') as f:
    steering_config = json.load(f)
BEST_LAYER = steering_config['best_layer']
v_steer = torch.load(v_steer_paths[0], map_location='cpu')
v_rand = torch.load(v_rand_paths[0], map_location='cpu')
print(f'✅ Layer: {BEST_LAYER} | v_steer: {v_steer.shape}')

In [ ]:
# Cell 6: Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'⌛ Loading {MODEL_NAME}...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
model.eval()
print('✅ Model loaded!')

In [ ]:
# Cell 7: Hook + Evaluation Engine (same as 3A)
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

class SteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=8, decay='hard'):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.decay = decay
        self.step_counter = 0
        self.handle = None
    
    def _eff_alpha(self, t):
        if self.K >= 999: return self.alpha
        if t >= self.K: return 0.0
        if self.decay == 'hard': return self.alpha
        elif self.decay == 'cosine':
            return self.alpha * 0.5 * (1.0 + math.cos(math.pi * t / self.K))
        elif self.decay == 'linear':
            return self.alpha * (1.0 - t / self.K)
        return self.alpha
    
    def hook_fn(self, module, inputs, output):
        a = self._eff_alpha(self.step_counter)
        if a > 0:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device).to(h.dtype)
                h[:, -1, :] = h[:, -1, :] + a * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device).to(output.dtype)
                output[:, -1, :] = output[:, -1, :] + a * v
        self.step_counter += 1
        return output
    
    def register(self, mdl):
        self.step_counter = 0
        self.handle = mdl.model.layers[self.layer_idx].register_forward_hook(self.hook_fn)
    def remove(self):
        if self.handle: self.handle.remove(); self.handle = None


def evaluate_condition(model, tokenizer, records, v_vector, alpha, K, decay,
                       max_new_tokens=80, name='', layer_idx=None):
    results = []
    t_start = time.time()
    for idx, rec in enumerate(tqdm(records, desc=name)):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        ref = rec['right_answer']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        hook = None
        if v_vector is not None and layer_idx is not None:
            hook = SteeringHook(layer_idx, v_vector, alpha=alpha, K=K, decay=decay)
            hook.register(model)
        
        torch.manual_seed(SEED + idx)
        t0 = time.time()
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=True, temperature=0.1, top_p=0.85)
        elapsed_ms = (time.time() - t0) * 1000
        if hook: hook.remove()
        
        gen_tokens = out_ids[0][inputs.input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        r_score = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100
        eos_id = tokenizer.eos_token_id
        hit_eos = bool(len(gen_tokens) > 0 and gen_tokens[-1].item() == eos_id)
        words = gen_text.split()
        if len(words) >= 4:
            ngrams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
            rep4 = 1.0 - len(set(ngrams))/len(ngrams) if ngrams else 0.0
        else: rep4 = 0.0
        
        results.append({
            'idx': idx, 'rouge_l': r_score, 'num_tokens': len(gen_tokens),
            'elapsed_ms': elapsed_ms, 'hit_eos': hit_eos, 'rep_4gram': rep4,
            'generated': gen_text, 'reference': ref, 'question': q,
            'category': rec.get('hallucination_type', 'unknown'),
        })
    avg_rl = np.mean([r['rouge_l'] for r in results])
    total_min = (time.time() - t_start) / 60
    print(f'  ✅ [{name}] {total_min:.1f}min | ROUGE-L: {avg_rl:.2f}%')
    return results

print('✅ Engine ready.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  ✅ CẤU HÌNH TỐI ƯU TỪ PHASE 3A (TỰ ĐỘNG ĐÃ LẮP VÀO)         ║
# ╚══════════════════════════════════════════════════════════════╝
BEST_ES_ALPHA = 18.0       # ← Tối ưu nhất từ Phase 3A
BEST_ES_K = 16             # ← Tối ưu nhất từ Phase 3A
BEST_ES_DECAY = 'linear'   # ← Tối ưu nhất từ Phase 3A

RUNNER_ES_ALPHA = 15.0     # ← Runner-up từ Phase 3A
RUNNER_ES_K = 16           # ← Runner-up từ Phase 3A
RUNNER_ES_DECAY = 'linear' # ← Runner-up từ Phase 3A

print(f'Best ES config:    α={BEST_ES_ALPHA}, K={BEST_ES_K}, decay={BEST_ES_DECAY}')
print(f'Runner-up config:  α={RUNNER_ES_ALPHA}, K={RUNNER_ES_K}, decay={RUNNER_ES_DECAY}')


In [ ]:
# Cell 9: BERTScore Computation
print('\n' + '='*70)
print('BERTSCORE COMPUTATION')
print('='*70)

try:
    from bert_score import score as bert_score_fn
    for cond, results in all_results.items():
        refs = [r['reference'] for r in results]
        hyps = [r['generated'] for r in results]
        print(f'  Computing BERTScore for {cond}...')
        P, R, F1 = bert_score_fn(hyps, refs, model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device='cuda')
        for r, bs in zip(results, F1.tolist()):
            r['bertscore_f1'] = bs
        print(f'    → {cond}: {np.mean(F1.tolist()):.4f}')
    print('✅ BERTScore done.')
except Exception as e:
    print(f'⚠️ BERTScore failed: {e}')

In [ ]:
# Cell 10: Category Breakdown + Statistical Test
print('\n' + '='*70)
print('CATEGORY BREAKDOWN')
print('='*70)

categories = sorted(set(r['category'] for r in all_results['baseline']))
print(f'{"Category":<35} {"Baseline":>8} {"ES Best":>8} {"Full":>8} {"Δ(ES-BL)":>9} {"Δ(ES-F)":>9}')
print('-'*80)
for cat in categories:
    bl = [r['rouge_l'] for r in all_results['baseline'] if r['category']==cat]
    es = [r['rouge_l'] for r in all_results['es_best'] if r['category']==cat]
    fs = [r['rouge_l'] for r in all_results['full_steering'] if r['category']==cat]
    if bl and es and fs:
        bl_m, es_m, fs_m = np.mean(bl), np.mean(es), np.mean(fs)
        print(f'{cat:<35} {bl_m:>7.2f}% {es_m:>7.2f}% {fs_m:>7.2f}% {es_m-bl_m:>+8.2f} {es_m-fs_m:>+8.2f}')

# Paired Bootstrap p-value
print('\n--- Paired Bootstrap Significance Test (10,000 resamples) ---')
bl_s = np.array([r['rouge_l'] for r in all_results['baseline']])
es_s = np.array([r['rouge_l'] for r in all_results['es_best']])
fs_s = np.array([r['rouge_l'] for r in all_results['full_steering']])

def boot_p(a, b, n=10000):
    diff = np.mean(a) - np.mean(b)
    cnt = sum(np.mean(a[np.random.randint(0,len(a),len(a))]) - 
              np.mean(b[np.random.randint(0,len(b),len(b))]) <= 0 for _ in range(n))
    return cnt/n

print(f'  ES Best vs Baseline:      Δ={np.mean(es_s)-np.mean(bl_s):+.2f}pp, p={boot_p(es_s,bl_s):.4f}')
print(f'  ES Best vs Full Steering: Δ={np.mean(es_s)-np.mean(fs_s):+.2f}pp, p={boot_p(es_s,fs_s):.4f}')
print(f'  Full vs Baseline:         Δ={np.mean(fs_s)-np.mean(bl_s):+.2f}pp, p={boot_p(fs_s,bl_s):.4f}')

In [ ]:
# Cell 11: Qualitative Examples
print('\n' + '='*70)
print('QUALITATIVE EXAMPLES (5 samples)')
print('='*70)

np.random.seed(42)
sample_ids = np.random.choice(len(all_results['baseline']), size=5, replace=False)

for i, idx in enumerate(sample_ids):
    bl = all_results['baseline'][idx]
    es = all_results['es_best'][idx]
    fs = all_results['full_steering'][idx]
    print(f'\n--- Example {i+1} [{bl["category"]}] ---')
    print(f'Q: {bl["question"][:120]}...')
    print(f'Ref: {bl["reference"][:120]}...')
    print(f'BL  (RL={bl["rouge_l"]:.1f}%): {bl["generated"][:150]}')
    print(f'ES  (RL={es["rouge_l"]:.1f}%): {es["generated"][:150]}')
    print(f'Full(RL={fs["rouge_l"]:.1f}%): {fs["generated"][:150]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  ✅ CẤU HÌNH TỐI ƯU TỪ PHASE 3A (TỰ ĐỘNG ĐÃ LẮP VÀO)         ║
# ╚══════════════════════════════════════════════════════════════╝
BEST_ES_ALPHA = 18.0       # ← Tối ưu nhất từ Phase 3A
BEST_ES_K = 16             # ← Tối ưu nhất từ Phase 3A
BEST_ES_DECAY = 'linear'   # ← Tối ưu nhất từ Phase 3A

RUNNER_ES_ALPHA = 15.0     # ← Runner-up từ Phase 3A
RUNNER_ES_K = 16           # ← Runner-up từ Phase 3A
RUNNER_ES_DECAY = 'linear' # ← Runner-up từ Phase 3A

print(f'Best ES config:    α={BEST_ES_ALPHA}, K={BEST_ES_K}, decay={BEST_ES_DECAY}')
print(f'Runner-up config:  α={RUNNER_ES_ALPHA}, K={RUNNER_ES_K}, decay={RUNNER_ES_DECAY}')
